# 02 — Construir Actual, Budget e Exportar para Excel
**Projeto:** budget-control-automation · **Etapa 2 de 3**

Aqui agregamos as vendas por mês/categoria (Actual), construímos um
Orçamento (Budget) com uma regra de negócio realista, e exportamos tudo
para um arquivo `.xlsx` de 3 abas — que será processado pela macro VBA.


## 1. Imports e carregamento

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

PROJECT_ROOT = Path.cwd().parent
RAW_PATH     = PROJECT_ROOT / "data" / "raw" / "Sample_Superstore.csv"
XLSX_PATH    = PROJECT_ROOT / "data" / "processed" / "budget_control_data.xlsx"

df = pd.read_csv(RAW_PATH, parse_dates=["Order Date"], encoding="latin1")
print(f"Carregado: {df.shape[0]:,} linhas")

## 2. Agregar o Actual (realizado) por mês e categoria

In [ ]:
# .dt.to_period("M") transforma uma data em "ano-mês" (ex: 2023-05)
# Isso agrupa todas as vendas do mesmo mês, independente do dia

df["month"] = df["Order Date"].dt.to_period("M")

actual = (
    df.groupby(["month", "Category"])
      .agg(actual_sales=("Sales", "sum"), actual_profit=("Profit", "sum"))
      .reset_index()
)
actual["month"] = actual["month"].astype(str)  # Period → texto, para salvar no Excel
actual = actual.sort_values(["Category", "month"]).reset_index(drop=True)

print(f"Linhas agregadas: {len(actual)}")
actual.head(8)

## 3. Construir o Budget — a lógica de negócio

**Por que média móvel de 3 meses?**
Orçamentos corporativos reais raramente são "chutados" — eles se baseiam
na tendência recente do próprio negócio, ajustada por uma meta de
crescimento definida pela liderança.

Aqui: `Budget do mês = média dos 3 meses anteriores × meta de crescimento da categoria`

Isso é exatamente o tipo de lógica usada em ferramentas de FP&A
(Financial Planning & Analysis) corporativas.


In [ ]:
# .shift(1) evita usar o próprio mês no cálculo da média (senão seria "trapaça")
# .rolling(3, min_periods=1) calcula a média dos 3 meses anteriores
#   min_periods=1 permite calcular mesmo com histórico incompleto (primeiros meses)

actual["rolling_avg"] = (
    actual.groupby("Category")["actual_sales"]
          .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

# Para o primeiro mês de cada categoria (sem histórico anterior), usamos a média geral
actual["rolling_avg"] = actual["rolling_avg"].fillna(
    actual.groupby("Category")["actual_sales"].transform("mean")
)

# Meta de crescimento definida por categoria (decisão de negócio, não um cálculo estatístico)
growth_target = {"Furniture": 1.05, "Office Supplies": 1.04, "Technology": 1.06}

np.random.seed(7)  # reprodutibilidade
actual["budget_sales"] = actual.apply(
    lambda r: round(
        r["rolling_avg"] * growth_target[r["Category"]] * np.random.normal(1.0, 0.04),
        2
    ),
    axis=1
)

print(actual[["month","Category","actual_sales","rolling_avg","budget_sales"]].head(10).to_string(index=False))

## 4. Prévia da variância (a macro VBA vai formalizar isso na planilha)

In [ ]:
actual["variance_pct"] = ((actual["actual_sales"] - actual["budget_sales"]) / actual["budget_sales"] * 100).round(1)

print("Prévia de Actual vs Budget:")
print(actual[["month","Category","actual_sales","budget_sales","variance_pct"]].head(10).to_string(index=False))
print(f"\nDesvio médio absoluto: {actual['variance_pct'].abs().mean():.1f}%")

## 5. Exportar para Excel — 3 abas (Actual, Budget, Variance_Report vazia)

In [ ]:
# A macro VBA vai ler as abas 'Actual' e 'Budget' e preencher 'Variance_Report'
# Por isso exportamos os dados brutos aqui — o CÁLCULO da variância final
# quem faz é a macro (é isso que demonstra a automação VBA)

wb = Workbook()
header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)

# Aba Actual
ws1 = wb.active
ws1.title = "Actual"
for col, h in enumerate(["Month","Category","Actual_Sales","Actual_Profit"], 1):
    c = ws1.cell(row=1, column=col, value=h)
    c.fill = header_fill; c.font = header_font
    c.alignment = Alignment(horizontal="center")
for i, row in actual.iterrows():
    ws1.cell(row=i+2, column=1, value=row["month"])
    ws1.cell(row=i+2, column=2, value=row["Category"])
    ws1.cell(row=i+2, column=3, value=round(row["actual_sales"],2))
    ws1.cell(row=i+2, column=4, value=round(row["actual_profit"],2))
for col in range(1,5):
    ws1.column_dimensions[get_column_letter(col)].width = 18

# Aba Budget
ws2 = wb.create_sheet("Budget")
for col, h in enumerate(["Month","Category","Budget_Sales"], 1):
    c = ws2.cell(row=1, column=col, value=h)
    c.fill = header_fill; c.font = header_font
    c.alignment = Alignment(horizontal="center")
for i, row in actual.iterrows():
    ws2.cell(row=i+2, column=1, value=row["month"])
    ws2.cell(row=i+2, column=2, value=row["Category"])
    ws2.cell(row=i+2, column=3, value=round(row["budget_sales"],2))
for col in range(1,4):
    ws2.column_dimensions[get_column_letter(col)].width = 18

# Aba vazia — a macro vai preencher
ws3 = wb.create_sheet("Variance_Report")
ws3.cell(row=1, column=1, value="Rode a macro RunBudgetControl (Alt+F8) para preencher esta aba")
ws3.cell(row=1, column=1).font = Font(italic=True, color="808080")

XLSX_PATH.parent.mkdir(parents=True, exist_ok=True)
wb.save(XLSX_PATH)
print(f"Workbook exportado: {XLSX_PATH}")
print(f"Linhas por aba: {len(actual)}")

## 6. Avançar

In [ ]:
print("=" * 50)
print("  Notebook 02 concluído.")
print("  PRÓXIMO PASSO: abra budget_control_data.xlsx no Excel")
print("  e siga o guia para instalar e rodar a macro VBA.")
print("=" * 50)